# P&ID Stage 1 — detector 1 lớp học trên trang ISO tự sinh (S1-ISO, PLAN_PHASE_C §2N)

Train detector **một lớp `symbol`** trên tile của **ISO-train** (625 trang ISO tự sinh từ khung xương PID2Graph, không trùng test / P4-val;
chỉ ảnh mẫu trong vocab — 19 mẫu ngoài vocab không bao giờ được vẽ) trộn với DigitizePID, khởi tạo từ `best.pt` của detector Stage 1
hiện tại. Rồi chấm **cũ và mới trên P4-val v2** (tập quyết định, không phân lớp, `agnostic_merge`). **Không** chạy test ở đây: nếu
detector mới được NHẬN (luật 6, áp ở máy bằng `scripts/s1iso_compare.py`), test v2 + e2e chạy ở notebook Stage 2 §19.

| Mục | Việc | Ra |
|---|---|---|
| 1–2 | GPU, Drive, code | — |
| 3 | giải nén tile §2N (50,248 train / 6,924 val) | `/content/s1iso/derived/tiles` |
| 4 | detector cũ (`DET_CKPT_OLD`) — **sửa đường dẫn** | — |
| 5–6 | train 20 epoch (resume nếu Colab ngắt) | `runs/s1iso/<ts>/train/weights/best.pt` |
| 7 | chốt checkpoint | `runs/encoders/det_s1iso.pt` |
| 8–9 | P4-val v2: cũ / mới, chấm không phân lớp | `runs/s1iso_csv/det_p4v2_{old,new}.json` |

**Trước khi chạy** (máy local):
- `cd data/s1iso_tiles && zip -r -0 ../s1iso_tiles.zip .` (~6.7 GB, ảnh JPEG nên `-0`) → upload thành `<DRIVE_ROOT>/data/s1iso_tiles.zip`.
- `cd data/iso_p4val_v2 && zip -r -0 iso_p4val_v2_canonical.zip canonical` → `<DRIVE_ROOT>/data/iso_p4val_v2_canonical.zip`.
- Commit + push nhánh (notebook, `configs/classes_symbol.yaml`, script §2N).

**Runtime**: GPU. **L4 khuyến nghị** (khoảng 2× T4, một phiên là xong; số compute unit gần như T4 vì chạy ngắn hơn); T4 chạy được
(~20 phút / epoch, cần resume qua 2 phiên); A100 nhanh nhất nhưng tốn unit hơn. Ultralytics tích luỹ gradient tới batch danh nghĩa 64,
nên `batch=-1` (tự chọn theo bộ nhớ GPU) không đổi công thức học giữa các GPU.

In [ ]:
# 1. GPU + Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import glob, hashlib, json, os, shutil, time
DRIVE_ROOT = next((d for d in ('/content/drive/MyDrive/pid', '/content/drive/MyDrive/PID_Recognition')
                   if os.path.isdir(f'{d}/data/canonical')), None)
assert DRIVE_ROOT, 'Không thấy <DRIVE_ROOT>/data/canonical — sửa DRIVE_ROOT bằng tay'
DATA_ROOT = f'{DRIVE_ROOT}/data'
RUNS_DIR  = f'{DRIVE_ROOT}/runs'
ENC_DIR   = f'{RUNS_DIR}/encoders'
S1_RUNS   = f'{RUNS_DIR}/s1iso'          # train runs (Drive: sống qua phiên)
KEEP      = f'{RUNS_DIR}/stage1_p4val_v2'  # detection P4-val v2 đã chạy (phiên sau không chạy lại)
CSV_DIR   = f'{RUNS_DIR}/s1iso_csv'      # metrics mang về máy
for d in (ENC_DIR, S1_RUNS, KEEP, CSV_DIR):
    os.makedirs(d, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)

In [ ]:
# 2. Code + deps
REPO_URL = 'https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git'   # <-- sửa
BRANCH   = 'feat/prepare-for-colab-stage-2'
%cd /content
![ -d pid ] || git clone -b $BRANCH $REPO_URL pid
%cd /content/pid
!git fetch -q && git checkout -q $BRANCH && git pull -q && git log --oneline -1
!pip install -q uv
!uv pip install --system -q ultralytics sahi ensemble-boxes albumentations timm hydra-core omegaconf \
    pycocotools opencv-python-headless pillow pandas scikit-learn matplotlib tqdm pyyaml
import sys
os.environ['PYTHONPATH'] = '/content/pid/src'
sys.path.insert(0, '/content/pid/src')
REPO_DATA = '/content/pid/data/raw'
S1_CLASSES = '/content/pid/configs/classes_symbol.yaml'    # 1 lớp `symbol` (§2N luật 3)
DET_CLASSES = f'{DATA_ROOT}/canonical/classes.yaml'        # 8 lớp coarse DigitizePID (detector cũ)
assert os.path.exists(S1_CLASSES), 'nhánh chưa có configs/classes_symbol.yaml: push lại'

In [ ]:
# 3. Tile §2N: giải nén về local (đọc từ Drive khi train sẽ rất chậm)
import yaml
S1_ZIP  = f'{DATA_ROOT}/s1iso_tiles.zip'
S1_ROOT = '/content/s1iso'                       # data_root khi train: detect_train đọc <root>/derived/tiles/data.yaml
TILES   = f'{S1_ROOT}/derived/tiles'
if not os.path.exists(f'{TILES}/tiles.csv'):
    assert os.path.exists(S1_ZIP), f'Chưa có {S1_ZIP}: ở local `cd data/s1iso_tiles && zip -r -0 ../s1iso_tiles.zip .` rồi upload'
    t0 = time.time()
    os.makedirs(TILES, exist_ok=True)
    !unzip -q -o "$S1_ZIP" -d "$TILES/"
    print(f'Giải nén sau {time.time() - t0:.0f}s')
n_train = len(glob.glob(f'{TILES}/images/train/*.jpg'))
n_val = len(glob.glob(f'{TILES}/images/val/*.jpg'))
assert (n_train, n_val) == (50248, 6924), (n_train, n_val, 'zip không phải bộ tile §2N?')
assert len(glob.glob(f'{TILES}/labels/train/*.txt')) == n_train
summary = json.load(open(f'{TILES}/summary.json'))
print(json.dumps(summary['pages']), '|', summary['boxes'], 'box')
# data.yaml với đường dẫn tuyệt đối của phiên này (bản trong zip ghi `path: .`)
yaml.safe_dump({'path': TILES, 'train': 'images/train', 'val': 'images/val', 'nc': 1, 'names': ['symbol']},
               open(f'{TILES}/data.yaml', 'w'), sort_keys=False)
print(open(f'{TILES}/data.yaml').read())

In [ ]:
# 4. Detector cũ (8 lớp DigitizePID) — điểm khởi tạo (§2N luật 5) và mốc so sánh trên P4-val v2.  SỬA DET_CKPT_OLD.
DET_CKPT_OLD = f'{RUNS_DIR}/2026-XX-XX_XX-XX-XX/train/weights/best.pt'   # <-- như DET_CKPT ở notebook Stage 2 mục 12a
if not os.path.exists(DET_CKPT_OLD):
    print('Ứng viên trên Drive:', sorted(glob.glob(f'{RUNS_DIR}/*/train/weights/best.pt')))
assert os.path.exists(DET_CKPT_OLD), f'Không thấy {DET_CKPT_OLD}: sửa DET_CKPT_OLD (ứng viên ở trên)'
from ultralytics import YOLO
old_names = list(YOLO(DET_CKPT_OLD).names.values())
assert len(old_names) == 8, old_names    # detector coarse DigitizePID
h = hashlib.sha256(open(DET_CKPT_OLD, 'rb').read()).hexdigest()
print('detector cũ:', old_names, '| sha256', h[:12])

In [ ]:
# 5. Train 20 epoch, 1 lớp, khởi tạo từ detector cũ (đầu ra 1 lớp khởi tạo lại; Ultralytics chép mọi lớp trùng hình)
# imgsz 1024 = tile; aug theo detector.aug của config (không xoay). best.pt = theo val YOLO (tile val, §2N luật 1).
EPOCHS = 20
!cd /content/pid && python src/run_pipeline.py detect --train \
    data_root=$S1_ROOT run_dir=$S1_RUNS detector.weights=$DET_CKPT_OLD \
    detector.epochs=$EPOCHS detector.batch=-1 detector.imgsz=1024 2>&1 | grep -v "^Warning" | tail -25

In [ ]:
# 6. Resume (phiên sau, khi Colab ngắt giữa chừng). Chạy 1–3 trước (cần tile local ở /content/s1iso).
last = sorted(glob.glob(f'{S1_RUNS}/*/train/weights/last.pt'), key=os.path.getmtime)
assert last, f'Chưa có run nào trong {S1_RUNS}: chạy mục 5'
RESUME_DIR = os.path.dirname(os.path.dirname(os.path.dirname(last[-1])))
print('resume', RESUME_DIR)
!cd /content/pid && python src/run_pipeline.py detect --train \
    data_root=$S1_ROOT run_dir=$S1_RUNS detector.resume=true detector.resume_dir=$RESUME_DIR 2>&1 | grep -v "^Warning" | tail -25

In [ ]:
# 7. Chốt checkpoint: best.pt của run mới nhất, 1 lớp `symbol`, đã đủ 20 epoch -> encoders/det_s1iso.pt
import csv as _csv
best = sorted(glob.glob(f'{S1_RUNS}/*/train/weights/best.pt'), key=os.path.getmtime)[-1]
run = os.path.dirname(os.path.dirname(best))
rows = [{k.strip(): v for k, v in r.items()} for r in _csv.DictReader(open(f'{run}/results.csv'))]   # cột cũ có khoảng trắng đầu
assert len(rows) == 20, f'{run}: mới {len(rows)} / 20 epoch — resume (mục 6)'
assert list(YOLO(best).names.values()) == ['symbol'], YOLO(best).names
DET_CKPT_NEW = f'{ENC_DIR}/det_s1iso.pt'
shutil.copy2(best, DET_CKPT_NEW)
fit = max(rows, key=lambda r: float(r['metrics/mAP50-95(B)'].strip() or 0))
print(f"{best} -> {DET_CKPT_NEW}\n  best epoch {fit['epoch'].strip()}: val mAP50 {float(fit['metrics/mAP50(B)']):.3f}"
      f"  mAP50-95 {float(fit['metrics/mAP50-95(B)']):.3f}  P {float(fit['metrics/precision(B)']):.3f}  R {float(fit['metrics/recall(B)']):.3f}")
print('sha256', hashlib.sha256(open(DET_CKPT_NEW, 'rb').read()).hexdigest()[:12])

**8–9. Quyết định trên P4-val v2** (§2N luật 6): detector cũ và mới trên 73 ảnh suy giảm của P4-val v2 (40 khung xương, kiểu v2: có
khung bản vẽ), cùng `agnostic_merge`, chấm `det_only` không phân lớp, nhóm theo R1a (`__global_lo__.pt`, như 12a). Không chọn gì khác ở
đây; luật áp ở máy: tải `runs/s1iso_csv/det_p4v2_{old,new}.json` về `runs/s1iso_csv/` rồi
`python scripts/s1iso_compare.py --out runs/phase_c/s1iso`.

In [ ]:
# 8. P4-val v2: giải nén, chạy detector cũ / mới (lưu detection lên Drive), chấm không phân lớp
assert 'DET_CKPT_NEW' in globals() or os.path.exists(f'{ENC_DIR}/det_s1iso.pt'), 'chạy mục 7 trước'
DET_CKPT_NEW = f'{ENC_DIR}/det_s1iso.pt'
P4V2_ZIP = f'{DATA_ROOT}/iso_p4val_v2_canonical.zip'
P4V2 = '/content/iso_p4val_v2'
if not os.path.exists(f'{P4V2}/canonical/annotations.coco.json'):
    assert os.path.exists(P4V2_ZIP), f'Chưa có {P4V2_ZIP}: `cd data/iso_p4val_v2 && zip -r -0 iso_p4val_v2_canonical.zip canonical` rồi upload'
    !mkdir -p "$P4V2" && unzip -q -o "$P4V2_ZIP" -d "$P4V2/"
coco = json.load(open(f'{P4V2}/canonical/annotations.coco.json'))
n_sym = sum(not a.get('iscrowd') for a in coco['annotations'])
assert len(coco['images']) == 73 and n_sym == 4832 and all(im['degraded'] for im in coco['images']), 'zip không phải P4-val v2?'
assert all('drawing_box' in im for im in coco['images']), 'P4-val v2 phải có khung bản vẽ (drawing_box)'
P4V2_DET = f'{P4V2}/derived/detections/test'
def restore(src, dst, n):
    assert len(glob.glob(f'{src}/*.txt')) == n, (src, len(glob.glob(f'{src}/*.txt')))
    shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
GROUPS = f'oneshot.encoder_ckpt={ENC_DIR}/__global_lo__.pt oneshot.vocab_csv={REPO_DATA}/iso_symbols/vocab.csv'
for tag, ckpt, classes in (('old', DET_CKPT_OLD, DET_CLASSES), ('new', DET_CKPT_NEW, S1_CLASSES)):
    keep = f'{KEEP}/detections_{tag}'
    if len(glob.glob(f'{keep}/*.txt')) != 73:
        shutil.rmtree(P4V2_DET, ignore_errors=True)
        !cd /content/pid && python src/run_pipeline.py detect --infer \
            data_root=$P4V2 run_dir=$RUNS_DIR/det_p4v2_$tag eval.split=test detector.ckpt=$ckpt \
            detector.label_level=coarse ++detector.classes_yaml=$classes ++detector.agnostic_merge=true 2>&1 | grep -v "^Warning" | tail -2
        restore(P4V2_DET, keep, 73)
    restore(keep, P4V2_DET, 73)
    print(f'===== Stage 1 {tag} trên P4-val v2')
    !cd /content/pid && python src/run_pipeline.py evaluate \
        data_root=$P4V2 run_dir=$RUNS_DIR/eval_det_p4v2_$tag eval.mode=det_only eval.split=test \
        detector.label_level=coarse ++eval.class_agnostic=true $GROUPS 2>&1 | grep -v "^Warning" | grep "detection"
    m = json.load(open(sorted(glob.glob(f'{RUNS_DIR}/eval_det_p4v2_{tag}/*/metrics.json'), key=os.path.getmtime)[-1]))
    assert m['detection']['class_agnostic'], 'phải chấm không phân lớp'
    json.dump(m['detection'], open(f'{CSV_DIR}/det_p4v2_{tag}.json', 'w'), indent=1)
print('đã chép:', sorted(os.listdir(CSV_DIR)))

In [ ]:
# 9. Bảng nhanh cũ | mới (P4-val v2, suy giảm). Quyết định: ở máy, scripts/s1iso_compare.py (CI theo tờ)
fmt = lambda v: '    -' if v is None else f'{v:5.3f}'
D = {t: json.load(open(f'{CSV_DIR}/det_p4v2_{t}.json'))['degraded'] for t in ('old', 'new')}
for t, b in D.items():
    print(f"{t:4s} P {fmt(b['precision'])}  R {fmt(b['recall'])}  ({b['n_predictions']} dự đoán, TP {b['tp']}, FP {b['fp']}, {b['n_gt']} GT)")
    print('     nhóm: ' + '  '.join(f"{k} {fmt(r['recall'])} ({r['tp']}/{r['n']})" for k, r in b.get('recall_by_group', {}).items()))
    ig = b.get('ignore')
    if ig:
        print('     trong vùng bỏ qua:', ig['predictions_by_kind'])
print(f'\nTải {CSV_DIR}/det_p4v2_old.json, det_p4v2_new.json về runs/s1iso_csv/ rồi chạy:')
print('  python scripts/s1iso_compare.py --out runs/phase_c/s1iso')